# Scatterbrain response evaluation

Paste the raw chat response(s) from the app into the **PASTE** cell, then run all cells.

- **Graph 1:** cosine similarity of the cited (relevant) chunks to the question, per query type.
- **Graph 2:** cosine similarity of the answer (and each answer sentence) to the citations.

How it works, so you do not have to paste anything else:
- The **question** is taken from the response's `history` (last user message). Its **query type** (factual / summary / comparison / list / other) is guessed from keywords.
- The **text of each citation** is looked up read-only in the app's SQLite database by `child_id`.
- Embeddings use Ollama `nomic-embed-text` from `.env`. If Ollama is not running, a clearly flagged offline stub is used so the graphs still render (stub numbers are word overlap only, not real quality).

In [ ]:
# Setup: dependency check, config, helpers.
import importlib, subprocess, sys

for module, pin in {"numpy": "numpy==2.5.2", "pandas": "pandas==2.3.3", "httpx": "httpx==0.28.1",
                    "dotenv": "python-dotenv==1.2.2", "matplotlib": "matplotlib==3.9.2"}.items():
    try:
        importlib.import_module(module)
    except ImportError:
        print(f"Installing {pin} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pin])

import hashlib, json, os, re, sqlite3
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import dotenv_values

REPO_ROOT = Path(r"c:\Users\ozzey\Documents\git-projects\scatterbrain")
_env = dotenv_values(REPO_ROOT / ".env")  # only the keys below are used; nothing is printed
OLLAMA_BASE_URL = (_env.get("OLLAMA_BASE_URL") or "http://localhost:11434").rstrip("/")
OLLAMA_EMBEDDING_MODEL = _env.get("OLLAMA_EMBEDDING_MODEL") or "nomic-embed-text"


def find_db():
    """Locate the app's SQLite database.

    Returns
    -------
    pathlib.Path
        First existing, non-empty candidate from ``SQLITE_DB_PATH``.
    """
    name = _env.get("SQLITE_DB_PATH") or "scatterbrain.db"
    for p in (REPO_ROOT / "backend" / name, REPO_ROOT / name, REPO_ROOT / "backend" / "scatterbrain.db"):
        if p.exists() and p.stat().st_size > 0:
            return p
    raise FileNotFoundError("Could not find the Scatterbrain database under backend/ or the repo root.")


DB_PATH = find_db()

try:
    USE_STUB = httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3.0).status_code != 200
except httpx.HTTPError:
    USE_STUB = True
if USE_STUB:
    print("!" * 70, "\nOLLAMA NOT REACHABLE -> using STUB embeddings (word overlap, NOT real).")
    print("Run `ollama serve` and `ollama pull nomic-embed-text`, then re-run for real scores.\n" + "!" * 70)
else:
    print(f"Using REAL Ollama embeddings: {OLLAMA_EMBEDDING_MODEL}")
print(f"Database: {DB_PATH.name}")

_CACHE = {}


def embed_text(text):
    """Embed text with Ollama (or the stub), cached.

    Parameters
    ----------
    text : str
        Text to embed.

    Returns
    -------
    numpy.ndarray
        Embedding vector.
    """
    if text not in _CACHE:
        if USE_STUB:
            vec = np.zeros(256)
            for tok in re.findall(r"\w+", text.lower()):
                vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % 256] += 1.0
        else:
            r = httpx.post(f"{OLLAMA_BASE_URL}/api/embeddings",
                           json={"model": OLLAMA_EMBEDDING_MODEL, "prompt": text[:8000]}, timeout=120.0)
            r.raise_for_status()
            vec = np.asarray(r.json()["embedding"], dtype=float)
        _CACHE[text] = vec
    return _CACHE[text]


def cosine_similarity(a, b):
    """Cosine similarity of two vectors (0.0 if either is all zeros)."""
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / d) if d else 0.0


def classify_query_type(query):
    """Guess query type from keywords: factual, summary, comparison, list or other."""
    q = query.lower()
    if re.search(r"summari[sz]e|overview|main themes", q):
        return "summary"
    if re.search(r"compare|difference|versus|\bvs\b", q):
        return "comparison"
    if re.search(r"\blist\b|\bwhich\b|name the", q):
        return "list"
    if re.search(r"how many|what is|what are|how much|who|when", q):
        return "factual"
    return "other"


def split_sentences(text):
    """Split text into sentences on ., ! or ?."""
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]


def chunk_text(child_id):
    """Fetch a chunk's text from the database (read-only), or None if missing."""
    con = sqlite3.connect(f"file:{DB_PATH.as_posix()}?mode=ro", uri=True)
    try:
        row = con.execute("SELECT content FROM document_chunks WHERE chunk_id = ?", (child_id,)).fetchone()
        if row is None:
            row = con.execute("SELECT source_text FROM document_parents WHERE parent_id = ?", (child_id,)).fetchone()
    finally:
        con.close()
    return row[0] if row and row[0] else None


def parse_response(raw, idx):
    """Turn one pasted ChatResponse into query, answer and citations.

    Parameters
    ----------
    raw : str or dict
        Raw ChatResponse JSON (string or already parsed).
    idx : int
        Position in the pasted list, used in error messages.

    Returns
    -------
    dict
        Keys: ``id, query, query_type, answer, citations``.
    """
    try:
        data = json.loads(raw) if isinstance(raw, str) else raw
    except json.JSONDecodeError as exc:
        raise ValueError(f"Response #{idx + 1}: not valid JSON ({exc})") from exc
    if not isinstance(data, dict) or not str(data.get("response", "")).strip():
        raise ValueError(f"Response #{idx + 1}: expected a JSON object with a 'response' field")
    users = [h.get("content", "") for h in data.get("history") or []
             if isinstance(h, dict) and h.get("role") == "user"]
    query = users[-1] if users else "(question not found in history)"
    return {"id": f"R{idx + 1}", "query": query, "query_type": classify_query_type(query),
            "answer": data["response"], "citations": data.get("citations") or []}

## Paste your response(s) here
One raw ChatResponse JSON per item. The sample below uses real chunks from your database so it runs as-is. Replace it with your own.

In [ ]:
# ===== PASTE YOUR RESPONSE(S) HERE =====
PASTED = [
    '''{
  "response": "GraphRAG builds a graph index of entities from the source documents. It then generates community summaries and combines partial answers into a final response.",
  "history": [{"role": "user", "content": "How does GraphRAG answer questions about a whole dataset?"}],
  "retrieved_chunks": 2,
  "citations": [
    {"document_id": "84840065-95a4-496a-b038-535b6dab2352", "child_id": "84840065-95a4-496a-b038-535b6dab2352_chunk_2", "filename": "graphrag.pdf", "label": "graphrag.pdf, chunk 2"},
    {"document_id": "84840065-95a4-496a-b038-535b6dab2352", "child_id": "84840065-95a4-496a-b038-535b6dab2352_chunk_1", "filename": "graphrag.pdf", "label": "graphrag.pdf, chunk 1"}
  ]
}''',
    '''{
  "response": "The paper proposes GraphRAG for query-focused summarization over large text corpora.",
  "history": [{"role": "user", "content": "Summarize the paper."}],
  "retrieved_chunks": 1,
  "citations": [
    {"document_id": "84840065-95a4-496a-b038-535b6dab2352", "child_id": "84840065-95a4-496a-b038-535b6dab2352_chunk_0", "filename": "graphrag.pdf", "label": "graphrag.pdf, chunk 0"}
  ]
}''',
]

In [ ]:
# Parse, look up citation text, score.
parsed, rows, cit_rows, heatmaps = [], [], [], []
for i, raw in enumerate(PASTED):
    try:
        parsed.append(parse_response(raw, i))
    except ValueError as err:
        print("SKIPPED ->", err)
if not parsed:
    raise SystemExit("No valid responses. Fix the PASTE cell and re-run.")

mode = "STUB embeddings (not real)" if USE_STUB else f"Ollama {OLLAMA_EMBEDDING_MODEL}"
for r in parsed:
    q_vec, a_vec, sents = embed_text(r["query"]), embed_text(r["answer"]), split_sentences(r["answer"])
    names, texts = [], []
    for n, c in enumerate(r["citations"], 1):
        cid = c.get("child_id")
        text = chunk_text(cid) if cid else None
        if text is None:
            print(f"{r['id']}: citation {n} ({cid}) not found in the database; skipped.")
            continue
        names.append(f"[{n}] {c.get('label') or c.get('filename') or cid}")
        texts.append(text)
    if not texts:
        print(f"{r['id']}: no citation text available, nothing to score.")
        continue
    mat = np.array([[cosine_similarity(embed_text(s), embed_text(t)) for t in texts] for s in sents])
    heatmaps.append((r, sents, names, mat))
    for j, (name, text) in enumerate(zip(names, texts)):
        rows.append({"response": r["id"], "query_type": r["query_type"], "citation": name,
                     "query_vs_chunk": cosine_similarity(q_vec, embed_text(text)),
                     "answer_vs_chunk": cosine_similarity(a_vec, embed_text(text)),
                     "best_sentence_vs_chunk": mat[:, j].max()})
df = pd.DataFrame(rows)
print("Embeddings:", mode)
print({r["id"]: (r["query_type"], r["query"][:60]) for r in parsed})

## Graph 1: cosine similarity of relevant chunks per query type

In [ ]:
types = sorted(df["query_type"].unique())
data = [df.loc[df["query_type"] == t, "query_vs_chunk"].values for t in types]
fig, ax = plt.subplots(figsize=(max(6, 2 * len(types) + 3), 5))
ax.boxplot(data, positions=range(len(types)), widths=0.5, showmeans=True)
rng = np.random.default_rng(0)
for k, d in enumerate(data):
    ax.scatter(k + rng.uniform(-0.12, 0.12, len(d)), d, zorder=3, alpha=0.8, label="cited chunk" if k == 0 else None)
ax.set_xticks(range(len(types)))
ax.set_xticklabels([f"{t}\n(n={len(d)})" for t, d in zip(types, data)])
ax.set_xlabel("Query type")
ax.set_ylabel("Cosine similarity (question vs cited chunk)")
ax.set_title(f"Relevant-chunk similarity per query type\n({mode})")
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Graph 2: similarity of the answer to its citations
Heatmap: each answer sentence vs each citation. Bars: whole answer vs each citation.

In [ ]:
ncols = min(2, len(heatmaps))
nrows = -(-len(heatmaps) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 3.2 * nrows + 1), squeeze=False)
for ax in axes.ravel():
    ax.set_visible(False)
for ax, (r, sents, names, mat) in zip(axes.ravel(), heatmaps):
    ax.set_visible(True)
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels([n[:28] for n in names], rotation=30, ha="right", fontsize=8)
    ax.set_yticks(range(len(sents)))
    ax.set_yticklabels([s[:40] + ("..." if len(s) > 40 else "") for s in sents], fontsize=8)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center", color="w" if mat[i, j] < 0.6 else "k", fontsize=8)
    ax.set_xlabel("Cited chunk")
    ax.set_ylabel("Answer sentence")
    ax.set_title(f"{r['id']} ({r['query_type']})", fontsize=10)
    fig.colorbar(im, ax=ax, label="Cosine similarity")
fig.suptitle(f"Answer sentences vs citations ({mode})")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(max(7, len(df) + 3), 5))
colors = dict(zip(types, plt.cm.tab10.colors))
ax.bar(range(len(df)), df["answer_vs_chunk"], color=[colors[t] for t in df["query_type"]], edgecolor="k")
ax.set_xticks(range(len(df)))
ax.set_xticklabels([f"{a}\n{b[:20]}" for a, b in zip(df["response"], df["citation"])], rotation=45, ha="right", fontsize=8)
ax.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()], [f"query type: {t}" for t in colors], loc="lower right")
ax.set_xlabel("Response / citation")
ax.set_ylabel("Cosine similarity (answer vs cited chunk)")
ax.set_title(f"Answer similarity to each citation ({mode})")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print("Embeddings:", mode)
display(df.groupby("query_type")[["query_vs_chunk", "answer_vs_chunk", "best_sentence_vs_chunk"]].mean().round(3))
display(df.groupby("response")[["query_vs_chunk", "answer_vs_chunk", "best_sentence_vs_chunk"]].mean().round(3))

**Reading the results**
- High question-vs-chunk similarity (about 0.7+ with `nomic-embed-text`) means the cited text is close to what was asked.
- A low row in the heatmap is a claim that none of the citations backs up.
- Cosine between a short answer and a long chunk is naturally lower, so compare relative values rather than treating them as absolute grades.
- "Relevant chunks" here are the **cited** chunks, because the response does not include the other retrieved chunks.